# Data Access

> Functionality to access Euclid calibrated data frames for specific targets and observation IDs.

In [ ]:
# | default_exp euclid.data_access

In [ ]:
# | export

from astropy import table
from astropy.table import Table
from astroquery.utils.tap.core import TapPlus
from getpass import getpass

import os

import numpy as np

In [ ]:
# | hide
from nbdev import show_doc

In [ ]:
# | export


class DataAccess:
    """Provides access to Euclid data."""

    def __init__(
        self,
        esa_username=None,  # ESA account username (prompts if not supplied)
        esa_password=None,  # ESA account password (prompts if not supplied)
        esac_server_url="https://easidr.esac.esa.int",  # ESA server (default is Q1)
        dry_run=False,  # if True, do not actually download files
    ):
        """Create an object for accessing data and log in to the ESA server."""
        if esa_username is None:
            self.esa_username = getpass(prompt="ESA User:")
        else:
            self.esa_username = esa_username
        if esa_password is None:
            self.esa_password = getpass(prompt="ESA Password:")
        else:
            self.esa_password = esa_password
        self.dry_run = dry_run
        self.tap = TapPlus(url=f"{esac_server_url}/tap-server", tap_context="tap")
        self.data_tap = TapPlus(url=f"{esac_server_url}/sas-dd", data_context="data")

    def tap_login(self):
        self.tap.login(user=self.esa_username, password=self.esa_password)

    def data_login(self):
        self.data_tap.login(user=self.esa_username, password=self.esa_password)

    def find_observations_for_target(
        self,
        ra,  # RA of the target, in decimal degrees
        dec,  # Dec of the target, in decimal degrees
        radius=1 / 60,  # radius of the target, in decimal degrees
        fully_contained=True,  # if False, the target region only needs to intersect with the observation footprint
    ):  # returns a list of observation_ids
        """Obtain a list of survey obs_ids for observations that entirely contain or intersect the specified target region."""
        criterion = "CONTAINS" if fully_contained else "INTERSECTS"
        query = f"""SELECT observation_stack.observation_id
                    FROM sedm.calibrated_frame
                    AS observation_stack
                    WHERE (product_type like '%Calibrated%')
                    AND (release_name not like 'CALBLOCK%')
                    AND (observation_stack.fov IS NOT NULL AND {criterion}(CIRCLE('ICRS',{ra},{dec},{radius}),observation_stack.fov)=1)
                    ORDER BY observation_id ASC"""
        self.tap_login()
        job = self.tap.launch_job(query)
        results = job.get_results()
        obs_ids = np.unique(list(results["observation_id"])).astype(int)
        return obs_ids

    def get_calibrated_files_for_observation(
        self,
        obs_id,  # observation_id for which to find files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
    ):  # returns a table of file information
        """Obtain file information for obs_id, optionally restricted by instrument or filter."""
        instrument_condition = (
            f"AND instrument_name = '{instrument}'" if instrument is not None else ""
        )
        filter_condition = f"AND filter_name = '{filter}'" if filter is not None else ""
        query = f"""SELECT observation_stack.observation_id, observation_stack.file_name,
                    observation_stack.instrument_name, observation_stack.filter_name, observation_stack.duration
                    FROM sedm.calibrated_frame
                    AS observation_stack
                    WHERE (product_type like '%Calibrated%')
                    {instrument_condition}
                    {filter_condition}
                    AND (observation_id = '{obs_id}')"""
        self.tap_login()
        job = self.tap.launch_job(query)
        file_info = job.get_results()
        return file_info

    def download_files(
        self,
        filenames,  #  list of filenames or Table containing "file_name" column
        outpath="./",  # the folder in which to save the downloaded files
        verbose=True,  # print information to the screen
    ):
        """Download multiple Euclid filenames to outpath."""
        if isinstance(filenames, Table):
            filenames = filenames["file_name"]
        if verbose:
            print(f"Downloading {len(filenames)} files to {outpath}")
        for fn in filenames:
            if verbose:
                print(f"Downloading {fn}")
            self.download_file(fn, outpath)

    def download_file(
        self,
        filename,  # the filename to download
        outpath="./",  # the folder in which to save the downloaded files
    ):
        """Download Euclid filename to outpath."""
        params_dict = dict(
            RETRIEVAL_TYPE="FILE", RELEASE="sedm", RETRIEVAL_ACCESS="DIRECT"
        )
        params_dict.update(FILE_NAME=filename)
        outpath = os.path.expanduser(outpath)
        outfn = os.path.join(outpath, filename)
        self.data_login()
        if not self.dry_run:
            self.data_tap.load_data(params_dict=params_dict, output_file=outfn)

    def download_calibrated_files_for_observation(
        self,
        obs_id,
        outpath="./",  # the folder in which to save the downloaded files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
        verbose=True,  # print information to the screen
    ):  #  returns a table of file information
        """Download all calibrated files for a Euclid observation, optionally restricted by instrument or filter."""
        file_info = self.get_calibrated_files_for_observation(
            obs_id, instrument=instrument, filter=filter
        )
        self.download_files(file_info, outpath=outpath, verbose=verbose)
        return file_info

    def download_calibrated_files_for_target(
        self,
        ra,  # RA of the target, in decimal degrees
        dec,  # Dec of the target, in decimal degrees
        radius=1 / 60,  # radius of the target, in decimal degrees
        fully_contained=True,  # if False, the target region only needs to intersect with the observation footprint
        outpath="./",  # the folder in which to save the downloaded files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
        verbose=True,  # print information to the screen
    ):  #  returns a table of file information
        """Download all calibrated files for Euclid observations covering a target, optionally restricted by instrument or filter."""
        file_info = []
        obs_ids = self.find_observations_for_target(
            ra, dec, radius, fully_contained=fully_contained
        )
        for obs_id in obs_ids:
            if verbose:
                print(f"Downloading files for observation id {obs_id}")
            obs_file_info = self.download_calibrated_files_for_observation(
                obs_id,
                outpath=outpath,
                instrument=instrument,
                filter=filter,
                verbose=verbose,
            )
            file_info.append(obs_file_info)
        if len(file_info) > 0:
            return table.vstack(file_info)

In [ ]:
show_doc(DataAccess.find_observations_for_target)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L48){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.find_observations_for_target

>      DataAccess.find_observations_for_target (ra, dec,
>                                               radius=0.016666666666666666,
>                                               fully_contained=True)

*Obtain a list of survey obs_ids for observations that entirely contain or intersect the specified target region.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| ra |  |  | RA of the target, in decimal degrees |
| dec |  |  | Dec of the target, in decimal degrees |
| radius | float | 0.016666666666666666 | radius of the target, in decimal degrees |
| fully_contained | bool | True | if False, the target region only needs to intersect with the observation footprint |

In [ ]:
show_doc(DataAccess.get_calibrated_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L70){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.get_calibrated_files_for_observation

>      DataAccess.get_calibrated_files_for_observation (obs_id, instrument=None,
>                                                       filter=None)

*Obtain file information for obs_id, optionally restricted by instrument or filter.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| obs_id |  |  | observation_id for which to find files |
| instrument | NoneType | None | None, NISP or VIS |
| filter | NoneType | None | None, VIS, NIR_Y, NIR_J or NIR_H |

In [ ]:
show_doc(DataAccess.download_calibrated_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L126){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.download_calibrated_files_for_observation

>      DataAccess.download_calibrated_files_for_observation (obs_id,
>                                                            outpath='./',
>                                                            instrument=None,
>                                                            filter=None,
>                                                            verbose=True)

*Download all calibrated files for a Euclid observation, optionally restricted by instrument or filter.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| obs_id |  |  |  |
| outpath | str | ./ | the folder in which to save the downloaded files |
| instrument | NoneType | None | None, NISP or VIS |
| filter | NoneType | None | None, VIS, NIR_Y, NIR_J or NIR_H |
| verbose | bool | True | print information to the screen |

In [ ]:
show_doc(DataAccess.download_calibrated_files_for_target)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L141){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.download_calibrated_files_for_target

>      DataAccess.download_calibrated_files_for_target (ra, dec,
>                                                       radius=0.016666666666666
>                                                       666,
>                                                       fully_contained=True,
>                                                       outpath='./',
>                                                       instrument=None,
>                                                       filter=None,
>                                                       verbose=True)

*Download all calibrated files for Euclid observations covering a target, optionally restricted by instrument or filter.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| ra |  |  | RA of the target, in decimal degrees |
| dec |  |  | Dec of the target, in decimal degrees |
| radius | float | 0.016666666666666666 | radius of the target, in decimal degrees |
| fully_contained | bool | True | if False, the target region only needs to intersect with the observation footprint |
| outpath | str | ./ | the folder in which to save the downloaded files |
| instrument | NoneType | None | None, NISP or VIS |
| filter | NoneType | None | None, VIS, NIR_Y, NIR_J or NIR_H |
| verbose | bool | True | print information to the screen |

### Example

The following demonstrates use of the DataAccess class. This would normally first be imported using `from nicl.euclid.data_access import DataAccess`.

To actually download files, remove `dry_run=True`.

In [ ]:
da = DataAccess(dry_run=True)

ESA User: ········
ESA Password: ········


In [ ]:
da.find_observations_for_target(ra=265.2125, dec=67.3972)

INFO: OK [astroquery.utils.tap.core]


array([2681])

In [ ]:
da.find_observations_for_target(
    ra=265.2125, dec=67.3972, radius=1.0, fully_contained=False
)

INFO: OK [astroquery.utils.tap.core]


array([2681, 2682, 2686, 2687, 2688, 2691])

In [ ]:
da.get_calibrated_files_for_observation(2681, instrument="NISP", filter="NIR_H")

INFO: OK [astroquery.utils.tap.core]


observation_id,file_name,instrument_name,filter_name,duration
str255,str255,str255,str255,float64
2681,EUC_NIR_W-CAL-IMAGE_H-2681-3_20240930T175003.801620Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-2_20240930T175003.453972Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-0_20240930T175003.578743Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-1_20240930T175003.573405Z.fits,NISP,NIR_H,87.2448


In [ ]:
da.download_calibrated_files_for_observation(
    2681, instrument="NISP", filter="NIR_H", outpath="~/data/euclid/test/"
)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


observation_id,file_name,instrument_name,filter_name,duration
str255,str255,str255,str255,float64
2681,EUC_NIR_W-CAL-IMAGE_H-2681-2_20240930T175003.453972Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-1_20240930T175003.573405Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-0_20240930T175003.578743Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-3_20240930T175003.801620Z.fits,NISP,NIR_H,87.2448


In [ ]:
da.download_calibrated_files_for_target(
    ra=265.2125, dec=67.3972, instrument="NISP", filter="NIR_H"
)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


observation_id,file_name,instrument_name,filter_name,duration
str255,str255,str255,str255,float64
2681,EUC_NIR_W-CAL-IMAGE_H-2681-0_20240930T175003.578743Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-2_20240930T175003.453972Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-3_20240930T175003.801620Z.fits,NISP,NIR_H,87.2448
2681,EUC_NIR_W-CAL-IMAGE_H-2681-1_20240930T175003.573405Z.fits,NISP,NIR_H,87.2448


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()